In [1]:
# Install required packages
!pip -q install groq tavily-python gradio python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00


In [2]:
import os
import getpass
import gradio as gr
from groq import Groq
from tavily import TavilyClient

# -----------------------------
# Ask for API Keys
# -----------------------------
GROQ_API_KEY = getpass.getpass("🔑 Enter your Groq API Key: ")

TAVILY_API_KEY = getpass.getpass("🔑 Enter your Tavily API Key: ")

# -----------------------------
# Initialize Clients
# -----------------------------
client = Groq(api_key=GROQ_API_KEY)

tavily = TavilyClient(api_key=TAVILY_API_KEY)

MODEL = "llama-3.3-70b-versatile"

print("✅ APIs Connected Successfully!")

🔑 Enter your Groq API Key: ··········
🔑 Enter your Tavily API Key: ··········
✅ APIs Connected Successfully!


In [4]:
def diagnose_crop(image, crop_name, symptoms):

    try:

        # -----------------------------
        # Input Validation
        # -----------------------------
        if not crop_name.strip():
            return "⚠️ Please enter the crop name."

        if not symptoms.strip():
            return "⚠️ Please enter the symptoms."

        # -----------------------------
        # Step 1: Disease Prediction
        # -----------------------------
        prediction = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": """
You are an expert agricultural scientist.

Predict ONLY:

1. Disease Name
2. One-line reason

Return ONLY these two.
"""
                },
                {
                    "role": "user",
                    "content": f"""
Crop: {crop_name}

Symptoms:
{symptoms}
"""
                }
            ]
        )

        disease_prediction = prediction.choices[0].message.content.strip()

        # -----------------------------
        # Step 2: Search Latest Medicines
        # -----------------------------
        search = tavily.search(
            query=f"{crop_name} {disease_prediction} medicines treatment purchase",
            max_results=5
        )

        context = ""

        for result in search["results"]:
            context += f"""
Title: {result['title']}
URL: {result['url']}
Content: {result['content']}

"""

        # -----------------------------
        # Step 3: Generate Final Report
        # -----------------------------
        report = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": """
You are an agriculture expert.

Generate a beautiful markdown report.

Use this exact format.

# 🌿 Crop

# 🦠 Disease

# 📝 Cause

# 💊 Top 3 Medicines

Mention:
• Medicine Name
• Purpose

# 🌱 Organic Remedy

# 🛡 Prevention Tips

Use bullet points.

# 🛒 Purchase Links

Mention trusted websites.

# 👨‍🌾 Farmer Advice

Simple beginner friendly language.
"""
                },
                {
                    "role": "user",
                    "content": f"""
Crop:
{crop_name}

Symptoms:
{symptoms}

Predicted Disease:
{disease_prediction}

Internet Search Results:
{context}
"""
                }
            ]
        )

        return report.choices[0].message.content

    except Exception as e:

        return f"""
# ❌ Error

Something went wrong.

### Details

```python
{e}
```
"""

In [15]:
css = """
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap');

:root{
    --bg:#F7F2EA;
    --card:#FFFDF8;
    --green:#5C8A57;
    --green-dark:#3E6B41;
    --green-light:#E8F5E9;
    --border:#E8DED1;
    --text:#2F3A2F;
    --sub:#77746E;
}

/* Background */

body{
    background:var(--bg)!important;
}

.gradio-container{
    max-width:1550px!important;
    width:96%!important;
    margin:auto!important;
    background:var(--bg)!important;
    font-family:'Poppins',sans-serif!important;
    padding:25px!important;
}

footer{
    display:none!important;
}

/* Header */

.header{
    text-align:center;
    margin-bottom:40px;
}

.title{
    font-size:56px;
    font-weight:700;
    color:var(--green-dark);
    margin-bottom:10px;
}

.subtitle{
    color:var(--sub);
    font-size:18px;
}

/* Cards */

.card{
    background:var(--card)!important;
    border-radius:28px!important;
    border:1px solid var(--border)!important;
    padding:30px!important;
    box-shadow:0 10px 25px rgba(0,0,0,.06)!important;
}

/* Remove dark containers */

.gr-group,
.gr-box,
.gr-panel,
.gr-form{
    background:transparent!important;
    border:none!important;
}

/* Upload */

.gr-image{
    border-radius:20px!important;
    overflow:hidden!important;
}

/* Inputs */

input,
textarea{

    background:#ffffff!important;

    color:#2F3A2F!important;

    border:2px solid var(--border)!important;

    border-radius:18px!important;

    padding:15px!important;

    font-size:16px!important;

    transition:.3s;

}

input::placeholder,
textarea::placeholder{

    color:#999!important;

    opacity:1!important;

}

input:focus,
textarea:focus{

    border:2px solid var(--green)!important;

    box-shadow:0 0 0 4px rgba(92,138,87,.15)!important;

}

/* Button */

button{

    width:100%!important;

    height:60px!important;

    background:linear-gradient(90deg,#6FA96C,#5C8A57)!important;

    color:white!important;

    border:none!important;

    border-radius:18px!important;

    font-size:19px!important;

    font-weight:600!important;

    transition:.3s;

}

button:hover{

    transform:translateY(-2px);

    background:linear-gradient(90deg,#5C8A57,#3E6B41)!important;

}

/* Markdown Report */

.result-card{

    background:#FFFDF8!important;

    border:1px solid var(--border)!important;

    border-radius:22px!important;

    padding:28px!important;

    color:#2F3A2F!important;

    line-height:1.8;

}

.result-card h1{

    color:#3E6B41!important;

    border-bottom:2px solid #E8DED1;

    padding-bottom:10px;

    margin-top:25px;

}

.result-card h2{

    color:#5C8A57!important;

}

.result-card h3{

    color:#6B8E23!important;

}

.result-card strong{

    color:#355E3B!important;

}

.result-card p{

    color:#444!important;

}

.result-card li{

    margin-bottom:8px;

}

.result-card a{

    color:#2E7D32!important;

    font-weight:600;

    text-decoration:none;

}

.result-card hr{

    border:0;

    border-top:1px solid #E8DED1;

    margin:20px 0;

}

/* Scrollbar */

::-webkit-scrollbar{

width:8px;

}

::-webkit-scrollbar-thumb{

background:#CABDAE;

border-radius:20px;

}

/* Mobile */

@media(max-width:900px){

.title{

font-size:38px;

}

.card{

padding:20px!important;

}

}
/* Markdown text */
.result-card p,
.result-card li{
    color:#2F3A2F !important;
    font-size:17px !important;
    font-weight:500 !important;
    line-height:1.9 !important;
}

/* Bullet points */
.result-card ul{
    color:#2F3A2F !important;
    padding-left:28px !important;
}

.result-card li::marker{
    color:#5C8A57 !important;
}

/* Headings */
.result-card h1,
.result-card h2,
.result-card h3{
    color:#355E3B !important;
    font-weight:700 !important;
}
"""

In [ ]:
# -----------------------------
# Premium Theme
# -----------------------------

theme = gr.themes.Soft(
    primary_hue="green",
    secondary_hue="emerald",
    neutral_hue="stone"
)

with gr.Blocks(
    theme=theme,
    css=css,
    title="🌿 CropCare AI"
) as demo:

    # -----------------------------
    # Header
    # -----------------------------

    gr.HTML("""
    <div class="header">
        <div class="title">🌿 CropCare AI</div>
        <div class="subtitle">
            Smart Crop Disease Detection & Medicine Recommendation
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):

        # ==========================================================
        # LEFT PANEL
        # ==========================================================

        with gr.Column(scale=4):

            with gr.Group(elem_classes="card"):

                gr.Markdown("## 📷 Upload Crop Image")

                image = gr.Image(
                    type="filepath",
                    show_label=False,
                    height=380,
                    container=False
                )

                gr.Markdown("### 🌾 Crop Name")

                crop = gr.Textbox(
                    placeholder="Example: Tomato",
                    show_label=False,
                    container=False
                )

                gr.Markdown("### 🍃 Symptoms")

                symptoms = gr.Textbox(
                    show_label=False,
                    container=False,
                    lines=7,
                    placeholder="""
Example:

• Yellow leaves
• Brown spots
• White fungus
• Fruits rotting
• Leaves curling
"""
                )

                diagnose_btn = gr.Button(
                    "🌿 Diagnose Disease",
                    variant="primary"
                )

        # ==========================================================
        # RIGHT PANEL
        # ==========================================================

        with gr.Column(scale=6):

            with gr.Group(elem_classes="card"):

                gr.HTML("""
                <h2 style="
                    color:#3E6B41;
                    font-size:34px;
                    margin-bottom:15px;
                ">
                🩺 AI Diagnosis Report
                </h2>
                """)

                output = gr.Markdown(
                    value="""
# 🌱 Welcome!

Upload a crop image and provide the symptoms.

---

### The AI will generate

🦠 Disease Name

📝 Disease Cause

💊 Top Medicines

🌿 Organic Treatment

🛡 Prevention Tips

🛒 Purchase Links

👨‍🌾 Farmer Advice

---

Click **🌿 Diagnose Disease** to begin.
""",
                    elem_classes="result-card"
                )

    # ==========================================================
    # BUTTON
    # ==========================================================

    diagnose_btn.click(
        fn=diagnose_crop,
        inputs=[
            image,
            crop,
            symptoms
        ],
        outputs=output
    )

demo.launch(debug=True)

/tmp/ipykernel_2000/2075745978.py:11: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4768464a6822293a4c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
